# ③ bge-m3 sentence alignment — Minh Mệnh Chính Yếu  ·  notebook P3 / P3

Reads NB2's `ocr_zips/<VOL>.zip` (corrected Hán + VI passthrough) and aligns the Việt and Hán
sentences with bge-m3 (dense DP + sparse lexical), runs step-4 char-validation and the visual QA
overlays, then pushes a final `output/<VOL>.zip` per volume.

**Run order:** NB1 (VI OCR) → NB2 (Hán OCR + consensus) → **this**. Needs `ocr_zips/vol1..6.zip` from NB2.


## 1. Dependencies (bge-m3 only — NO surya/paddle/qwen)  →  then **Runtime ▸ Restart**

In [ ]:
# bge-m3 alignment only. NO surya, NO paddle, NO qwen — Hán is already OCR'd & corrected by notebook 2.
!pip -q install PyMuPDF opencv-python-headless pandas openpyxl tqdm huggingface_hub
!pip -q install FlagEmbedding                   # bge-m3 alignment (pulls torch + transformers)
print('\n>>> Now: Runtime ▸ Restart session, then run VERIFY.')


### Verify torch GPU (after restart)

In [ ]:
import numpy, torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Mount Drive + project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/minh_menh'
MMCY  = '/content/mmcy'                       # LOCAL work root (same as notebooks 1 & 2)
os.environ['MMCY_ROOT'] = MMCY
%cd $DRIVE
!mkdir -p {MMCY}/out
!rsync -a {DRIVE}/assets/ {MMCY}/assets/      # dicts (local, for build_dicts + step4 char-val)
print('code:', DRIVE, '| work root (local):', MMCY)


## 3. Build dictionaries (one-time; needed for step-4 char validation)

In [ ]:
!python -m pipeline.build_dicts


## 4. Run ALL 6 volumes — bge alignment, per-vol push to Drive

In [ ]:
# ── Setup for the 6-vol loop: load BGE model ONCE, define align helpers ──
from pathlib import Path
import numpy as np, pandas as pd
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm
from pipeline import config
from pipeline.common import read_jsonl, write_jsonl
from pipeline.step4_align import filter_vi_sentences, lexical_sim

VOLS = ["vol1", "vol2", "vol3", "vol4", "vol5", "vol6"]   # edit to run a subset

ALIGN = dict(config.ALIGN)
BGE_SIM_THRESHOLD = 0.50      # bge cosines ~0.45-0.66 (median ~0.57), not LaBSE's ~0-0.35
BGE_SUSPECT_DENSE = 0.45
ALIGN["sim_threshold"] = BGE_SIM_THRESHOLD
BATCH = 256

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)   # load ONCE, reuse every vol

def _merge_sparse(weights, idxs):
    m = {}
    for k in idxs:
        for t, w in weights[k].items(): m[t] = m.get(t, 0.0) + float(w)
    return m
def _group_unit(emb, k):
    n, d = emb.shape
    pref = np.zeros((n+1, d), "float64"); np.cumsum(emb, axis=0, out=pref[1:])
    s = pref[k:] - pref[:-k]
    nrm = np.linalg.norm(s, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
    return (s/nrm).astype("float32")
def align_bge(han, vie):
    H, V = len(han), len(vie)
    if not H or not V: return []
    sims = {(dh,dv): _group_unit(h_dense,dh) @ _group_unit(v_dense,dv).T
            for dh,dv in ALIGN["merge_modes"] if dh and dv}
    NEG=-1e9; dp=[[NEG]*(V+1) for _ in range(H+1)]; back=[[None]*(V+1) for _ in range(H+1)]
    dp[0][0]=0.0; thr=ALIGN["sim_threshold"]; modes=ALIGN["merge_modes"]
    for i in tqdm(range(H+1), desc="bge DP align", unit="row"):
        for j in range(V+1):
            base=dp[i][j]
            if base<=NEG: continue
            for dh,dv in modes:
                ni,nj=i+dh,j+dv
                if ni>H or nj>V or (dh==0 and dv==0): continue
                s=float(sims[(dh,dv)][i,j]) if (dh and dv) else 0.0
                gain=s if s>=thr else (s-1.0)
                if base+gain>dp[ni][nj]:
                    dp[ni][nj]=base+gain; back[ni][nj]=(i,j,dh,dv,round(s,3))
    pairs,i,j=[],H,V
    while (i,j)!=(0,0):
        prev=back[i][j]
        if prev is None: break
        pi,pj,dh,dv,s=prev
        am=" ".join(han[k]["am_han_viet"] for k in range(pi,pi+dh))
        vt=" ".join(vie[k]["text"] for k in range(pj,pj+dv))
        lex=round(lexical_sim(am,vt),3)
        bge_sp=round(float(model.compute_lexical_matching_score(
            _merge_sparse(h_sparse,range(pi,pi+dh)),
            _merge_sparse(v_sparse,range(pj,pj+dv)))),3) if (dh and dv) else 0.0
        pairs.append({"han_ids":[han[k]["id"] for k in range(pi,pi+dh)],
            "vie_ids":[vie[k]["id"] for k in range(pj,pj+dv)],
            "sinonom":"".join(han[k]["sinonom"] for k in range(pi,pi+dh)),
            "am_han_viet":am,"vietnamese":vt,"similarity":s,"lexical":lex,"bge_sparse":bge_sp,
            "suspect":bool(s<BGE_SUSPECT_DENSE and lex<ALIGN["suspect_lexical"])})
        i,j=pi,pj
    pairs.reverse(); return pairs

print("BGE model loaded; will process:", VOLS)

In [ ]:
# ── Align every volume; push EACH vol's output zip to Drive the moment it finishes ──
# Reads NB2 hand-off ocr_zips/<VOL>.zip (corrected Hán + VI passthrough + han_review/
# reviews), runs bge-m3 alignment + Excel export + visual QA overlays (P3 is alignment-only:
# the VI/Hán review queues are built in P1/P2), stages one folder, pushes output/<VOL>.zip.
import shutil, traceback, time
!mkdir -p {DRIVE}/output
done, failed = [], []
for idx, VOL in enumerate(VOLS):
    CHAPTER = idx + 1
    print(f"\n{'='*64}\n#### {VOL}  (chapter {CHAPTER})  ####\n{'='*64}")
    t0 = time.time()
    try:
        # ── hand-off from NB2: ONE zip -> unzip onto LOCAL ssd (Drive FUSE corrupts) ──
        IN_DIR = config.OUT_DIR / VOL
        !cp -f {DRIVE}/ocr_zips/{VOL}.zip {MMCY}/out/
        !cd {MMCY}/out && unzip -qo {VOL}.zip
        assert (IN_DIR / "han_sentences.jsonl").exists(), \
            f"{IN_DIR}/han_sentences.jsonl missing — run NB2 first (writes ocr_zips/{VOL}.zip)"
        assert (IN_DIR / "vi_sentences.jsonl").exists(), \
            f"{IN_DIR}/vi_sentences.jsonl missing — NB1 step2 must finish first"

        han_sents = list(read_jsonl(IN_DIR / "han_sentences.jsonl"))
        vie_sents = list(read_jsonl(IN_DIR / "vi_sentences.jsonl"))

        # ── bge-m3 alignment on the CORRECTED Hán -> output_bge/<VOL>/ ──
        OUT_DIR = config.ROOT / "output_bge" / VOL; OUT_DIR.mkdir(parents=True, exist_ok=True)
        vie_kept, n_drop = filter_vi_sentences(vie_sents, ALIGN)
        print(f"{len(han_sents)} Hán | {len(vie_sents)} VI -> kept {len(vie_kept)}, dropped {n_drop}")
        h_dense = model.encode([h["sinonom"] for h in han_sents], batch_size=BATCH,
                               return_dense=True, return_sparse=False)["dense_vecs"]
        v_out   = model.encode([v["text"] for v in vie_kept], batch_size=BATCH,
                               return_dense=True, return_sparse=True)
        v_dense, v_sparse = v_out["dense_vecs"], v_out["lexical_weights"]
        h_sparse = model.encode([h["am_han_viet"] for h in han_sents], batch_size=BATCH,
                                return_dense=False, return_sparse=True)["lexical_weights"]
        h_dense = np.asarray(h_dense, "float32"); v_dense = np.asarray(v_dense, "float32")
        print("dense shapes:", h_dense.shape, v_dense.shape)

        pairs = align_bge(han_sents, vie_kept)
        sp = np.array([p["bge_sparse"] for p in pairs if p["vietnamese"] and p["bge_sparse"]>0])
        SPARSE_THR = round(float(np.percentile(sp, 10)), 3) if len(sp) else 0.0
        for p in pairs:
            p["suspect_bge"] = bool(p["similarity"]<BGE_SUSPECT_DENSE and p["bge_sparse"]<SPARSE_THR)
        print(f"{len(pairs)} pairs | suspect={sum(p['suspect'] for p in pairs)} "
              f"| suspect_bge={sum(p['suspect_bge'] for p in pairs)} | SPARSE_THR={SPARSE_THR}")

        # write output_bge/<VOL>/ (jsonl + Excel)
        write_jsonl(OUT_DIR / "alignment.jsonl", pairs)
        df = pd.DataFrame([{
            "Hán IDs": ", ".join(p["han_ids"]), "Việt IDs": ", ".join(p["vie_ids"]),
            "SinoNom": p["sinonom"], "Âm Hán Việt": p["am_han_viet"],
            "Nghĩa thuần Việt": p["vietnamese"], "Dense (bge)": p["similarity"],
            "Lexical (Jaccard)": p["lexical"], "Sparse (bge)": p["bge_sparse"],
            "Nghi lệch (Jaccard)": "x" if p["suspect"] else "",
            "Nghi lệch (bge)": "x" if p.get("suspect_bge") else "",
        } for p in pairs])
        s = config.ID_SCHEMA
        xlsx = OUT_DIR / f"{s['domain']}{s['subdomain']}{s['genre']}_{s['file_id']}_alignment_bge.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
            df.to_excel(xl, sheet_name="sentence_alignment", index=False)
        print("wrote", OUT_DIR / "alignment.jsonl", "and", xlsx.name)

        # ── Excel export (box + sentence sheets) + visual QA overlays ──
        # P3 is alignment-only: VI review (P1) + Hán char-val/review (P2) already ran.
        !python -m pipeline.step4_align --vol {VOL}
        !python -m pipeline.draw_boxes --vol {VOL} --align {MMCY}/output_bge/{VOL}/alignment.jsonl

        # ── stage ONE folder, then push THIS vol's zip to Drive now (FUSE-safe) ──
        STAGE = config.ROOT / 'output' / VOL
        if STAGE.exists(): shutil.rmtree(STAGE)
        STAGE.mkdir(parents=True)
        src = config.OUT_DIR / VOL
        bge = config.ROOT / 'output_bge' / VOL
        for d in ['pages_han_boxed','pages_han_aligned','pages_vi_boxed','pages_vi_aligned']:
            if (src/d).exists(): shutil.copytree(src/d, STAGE/d)
        for f in ['han_boxes.jsonl','han_sentences.jsonl','vi_boxes.jsonl','vi_sentences.jsonl',
                  'vi_review.jsonl','han_review.jsonl','oov_vocab.jsonl',
                  'split_manifest.json','consensus.jsonl','review_queue.jsonl','consensus_metrics.json']:
            if (src/f).exists(): shutil.copy2(src/f, STAGE/f)
        shutil.copy2(bge/'alignment.jsonl', STAGE/'alignment.jsonl')   # BGE, not step4's 0-pairs
        for x in bge.glob('*_alignment_bge.xlsx'): shutil.copy2(x, STAGE/x.name)
        !cd {MMCY}/output && zip -qr {DRIVE}/output/{VOL}.zip {VOL}
        !ls -lh {DRIVE}/output/{VOL}.zip
        print(f"✅ {VOL} DONE in {(time.time()-t0)/60:.1f} min -> Drive output/{VOL}.zip")
        done.append(VOL)
    except Exception as e:
        traceback.print_exc()
        print(f"❌ {VOL} FAILED: {e} — continuing to next vol")
        failed.append(VOL)

print(f"\n{'='*64}\nSUMMARY  done={done}  failed={failed}")
